In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

# -----------------------------
# Data Augmentation
# -----------------------------
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    transforms.RandomErasing(p=0.2)
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=False, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=False, transform=transform_test)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)

# -----------------------------
# ADMM Layers
# -----------------------------
class ADMMConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, rho=1e-4):
        super(ADMMConv2d, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=True)
        self.rho = rho
        self.register_buffer("Z", torch.zeros_like(self.conv.weight))
        self.register_buffer("U", torch.zeros_like(self.conv.weight))

    def forward(self, x):
        return self.conv(x)

    def admm_loss(self):
        return (self.rho / 2) * torch.norm(self.conv.weight - self.Z + self.U) ** 2

    def update_ZU(self):
        with torch.no_grad():
            temp = self.conv.weight + self.U
            threshold = self.rho
            self.Z = torch.sign(temp) * torch.clamp(torch.abs(temp) - threshold, min=0)
            self.U = self.U + (self.conv.weight - self.Z)


class ADMMLinear(nn.Module):
    def __init__(self, in_features, out_features, rho=1e-4):
        super(ADMMLinear, self).__init__()
        self.fc = nn.Linear(in_features, out_features)
        self.rho = rho
        self.register_buffer("Z", torch.zeros_like(self.fc.weight))
        self.register_buffer("U", torch.zeros_like(self.fc.weight))

    def forward(self, x):
        return self.fc(x)

    def admm_loss(self):
        return (self.rho / 2) * torch.norm(self.fc.weight - self.Z + self.U) ** 2

    def update_ZU(self):
        with torch.no_grad():
            temp = self.fc.weight + self.U
            threshold = self.rho
            self.Z = torch.sign(temp) * torch.clamp(torch.abs(temp) - threshold, min=0)
            self.U = self.U + (self.fc.weight - self.Z)


# -----------------------------
# CNN with ADMM
# -----------------------------
class CNN_ADMM(nn.Module):
    def __init__(self, rho=1e-4):
        super(CNN_ADMM, self).__init__()
        self.conv = nn.Sequential(
            ADMMConv2d(3, 64, 3, padding=1, rho=rho), nn.BatchNorm2d(64), nn.ReLU(),
            ADMMConv2d(64, 64, 3, padding=1, rho=rho), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 16x16

            ADMMConv2d(64, 128, 3, padding=1, rho=rho), nn.BatchNorm2d(128), nn.ReLU(),
            ADMMConv2d(128, 128, 3, padding=1, rho=rho), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 8x8

            ADMMConv2d(128, 256, 3, padding=1, rho=rho), nn.BatchNorm2d(256), nn.ReLU(),
            ADMMConv2d(256, 256, 3, padding=1, rho=rho), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 4x4
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            ADMMLinear(256 * 4 * 4, 512, rho=rho), nn.ReLU(), nn.Dropout(0.4),
            ADMMLinear(512, 10, rho=rho)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

    def admm_loss(self):
        penalty = 0
        for module in self.modules():
            if isinstance(module, (ADMMConv2d, ADMMLinear)):
                penalty += module.admm_loss()
        return penalty

    def update_ZU(self):
        for module in self.modules():
            if isinstance(module, (ADMMConv2d, ADMMLinear)):
                module.update_ZU()


# -----------------------------
# Setup
# -----------------------------
model = CNN_ADMM(rho=1e-4).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=70)

# -----------------------------
# Warm-up Training
# -----------------------------
print("Warm-up phase...")
for epoch in range(20):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Warm-up Epoch [{epoch+1}/20] Loss: {total_loss/len(train_loader):.4f}")

# -----------------------------
# ADMM Pruning Phase
# -----------------------------
print("ADMM pruning phase...")
for epoch in range(60):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels) + model.admm_loss()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Update Z and U
    model.update_ZU()
    scheduler.step()
    print(f"ADMM Epoch [{epoch+1}/60] Loss: {total_loss/len(train_loader):.4f}")

# -----------------------------
# Fine-tuning
# -----------------------------
print("Fine-tuning pruned model...")
for epoch in range(20):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Fine-tune Epoch [{epoch+1}/20] Loss: {total_loss/len(train_loader):.4f}")

# -----------------------------
# Evaluation
# -----------------------------
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

acc = 100 * correct / total
print(f"Final Accuracy after ADMM pruning + fine-tuning: {acc:.2f}%")

# Save model
torch.save(model.state_dict(), 'cnn_admm_pruned_improved.pth')


Running on: cuda
Warm-up phase...
Warm-up Epoch [1/20] Loss: 1.9047
Warm-up Epoch [2/20] Loss: 1.5343
Warm-up Epoch [3/20] Loss: 1.3746
Warm-up Epoch [4/20] Loss: 1.2751
Warm-up Epoch [5/20] Loss: 1.1939
Warm-up Epoch [6/20] Loss: 1.1315
Warm-up Epoch [7/20] Loss: 1.0864
Warm-up Epoch [8/20] Loss: 1.0426
Warm-up Epoch [9/20] Loss: 1.0111
Warm-up Epoch [10/20] Loss: 0.9803
Warm-up Epoch [11/20] Loss: 0.9545
Warm-up Epoch [12/20] Loss: 0.9333
Warm-up Epoch [13/20] Loss: 0.9123
Warm-up Epoch [14/20] Loss: 0.8956
Warm-up Epoch [15/20] Loss: 0.8788
Warm-up Epoch [16/20] Loss: 0.8625
Warm-up Epoch [17/20] Loss: 0.8535
Warm-up Epoch [18/20] Loss: 0.8331
Warm-up Epoch [19/20] Loss: 0.8280
Warm-up Epoch [20/20] Loss: 0.8137
ADMM pruning phase...
ADMM Epoch [1/60] Loss: 1.1562
ADMM Epoch [2/60] Loss: 0.7978
ADMM Epoch [3/60] Loss: 0.7837
ADMM Epoch [4/60] Loss: 0.7690
ADMM Epoch [5/60] Loss: 0.7621
ADMM Epoch [6/60] Loss: 0.7499
ADMM Epoch [7/60] Loss: 0.7366
ADMM Epoch [8/60] Loss: 0.7305
ADMM 

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

# -----------------------------
# Data Augmentation
# -----------------------------
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    transforms.RandomErasing(p=0.2)
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=False, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=False, transform=transform_test)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)

# -----------------------------
# ADMM Layers
# -----------------------------
class ADMMConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, rho=1e-4):
        super(ADMMConv2d, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=True)
        self.rho = rho
        self.register_buffer("Z", torch.zeros_like(self.conv.weight))
        self.register_buffer("U", torch.zeros_like(self.conv.weight))

    def forward(self, x):
        return self.conv(x)

    def admm_loss(self):
        return (self.rho / 2) * torch.norm(self.conv.weight - self.Z + self.U) ** 2

    def update_ZU(self):
        with torch.no_grad():
            temp = self.conv.weight + self.U
            threshold = self.rho
            self.Z = torch.sign(temp) * torch.clamp(torch.abs(temp) - threshold, min=0)
            self.U = self.U + (self.conv.weight - self.Z)


class ADMMLinear(nn.Module):
    def __init__(self, in_features, out_features, rho=1e-4):
        super(ADMMLinear, self).__init__()
        self.fc = nn.Linear(in_features, out_features)
        self.rho = rho
        self.register_buffer("Z", torch.zeros_like(self.fc.weight))
        self.register_buffer("U", torch.zeros_like(self.fc.weight))

    def forward(self, x):
        return self.fc(x)

    def admm_loss(self):
        return (self.rho / 2) * torch.norm(self.fc.weight - self.Z + self.U) ** 2

    def update_ZU(self):
        with torch.no_grad():
            temp = self.fc.weight + self.U
            threshold = self.rho
            self.Z = torch.sign(temp) * torch.clamp(torch.abs(temp) - threshold, min=0)
            self.U = self.U + (self.fc.weight - self.Z)


# -----------------------------
# CNN with ADMM
# -----------------------------
class CNN_ADMM(nn.Module):
    def __init__(self, rho=1e-4):
        super(CNN_ADMM, self).__init__()
        self.conv = nn.Sequential(
            ADMMConv2d(3, 64, 3, padding=1, rho=rho), nn.BatchNorm2d(64), nn.ReLU(),
            ADMMConv2d(64, 64, 3, padding=1, rho=rho), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 16x16

            ADMMConv2d(64, 128, 3, padding=1, rho=rho), nn.BatchNorm2d(128), nn.ReLU(),
            ADMMConv2d(128, 128, 3, padding=1, rho=rho), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 8x8

            ADMMConv2d(128, 256, 3, padding=1, rho=rho), nn.BatchNorm2d(256), nn.ReLU(),
            ADMMConv2d(256, 256, 3, padding=1, rho=rho), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 4x4
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            ADMMLinear(256 * 4 * 4, 512, rho=rho), nn.ReLU(), nn.Dropout(0.4),
            ADMMLinear(512, 10, rho=rho)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

    def admm_loss(self):
        penalty = 0
        for module in self.modules():
            if isinstance(module, (ADMMConv2d, ADMMLinear)):
                penalty += module.admm_loss()
        return penalty

    def update_ZU(self):
        for module in self.modules():
            if isinstance(module, (ADMMConv2d, ADMMLinear)):
                module.update_ZU()


# -----------------------------
# Setup
# -----------------------------
model = CNN_ADMM(rho=1e-4).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=70)

# -----------------------------
# ADMM Pruning Phase
# -----------------------------
print("ADMM pruning phase...")
for epoch in range(50):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels) + model.admm_loss()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Update Z and U
    model.update_ZU()
    scheduler.step()
    print(f"ADMM Epoch [{epoch+1}/50] Loss: {total_loss/len(train_loader):.4f}")

# -----------------------------
# Evaluation
# -----------------------------
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

acc = 100 * correct / total
print(f"Final Accuracy after ADMM pruning + fine-tuning: {acc:.2f}%")

# Save model
torch.save(model.state_dict(), 'cnn_admm_pruned_improved.pth')


Running on: cuda
ADMM pruning phase...
ADMM Epoch [1/50] Loss: 1.8966
ADMM Epoch [2/50] Loss: 1.4998
ADMM Epoch [3/50] Loss: 1.3319
ADMM Epoch [4/50] Loss: 1.2302
ADMM Epoch [5/50] Loss: 1.1562
ADMM Epoch [6/50] Loss: 1.0945
ADMM Epoch [7/50] Loss: 1.0557
ADMM Epoch [8/50] Loss: 1.0206
ADMM Epoch [9/50] Loss: 0.9884
ADMM Epoch [10/50] Loss: 0.9593
ADMM Epoch [11/50] Loss: 0.9365
ADMM Epoch [12/50] Loss: 0.9167
ADMM Epoch [13/50] Loss: 0.8997
ADMM Epoch [14/50] Loss: 0.8812
ADMM Epoch [15/50] Loss: 0.8638
ADMM Epoch [16/50] Loss: 0.8528
ADMM Epoch [17/50] Loss: 0.8374
ADMM Epoch [18/50] Loss: 0.8242
ADMM Epoch [19/50] Loss: 0.8106
ADMM Epoch [20/50] Loss: 0.7994
ADMM Epoch [21/50] Loss: 0.7868
ADMM Epoch [22/50] Loss: 0.7807
ADMM Epoch [23/50] Loss: 0.7699
ADMM Epoch [24/50] Loss: 0.7565
ADMM Epoch [25/50] Loss: 0.7523
ADMM Epoch [26/50] Loss: 0.7417
ADMM Epoch [27/50] Loss: 0.7341
ADMM Epoch [28/50] Loss: 0.7261
ADMM Epoch [29/50] Loss: 0.7200
ADMM Epoch [30/50] Loss: 0.7141
ADMM Epoch